In [1]:
import numpy as np
import statsmodels.api as sm

In [2]:
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)

# Weekly Dataset

In [3]:
weekly = load_data("Weekly")
weekly

,Year,Lag1,Lag2,Lag3,Lag4,Lag5,Volume,Today,Direction
0,1990,0.816,1.572,-3.936,-0.229,-3.484,0.154976,-0.270,Down
1,1990,-0.270,0.816,1.572,-3.936,-0.229,0.148574,-2.576,Down
2,1990,-2.576,-0.270,0.816,1.572,-3.936,0.159837,3.514,Up
3,1990,3.514,-2.576,-0.270,0.816,1.572,0.161630,0.712,Up
4,1990,0.712,3.514,-2.576,-0.270,0.816,0.153728,1.178,Up
...,...,...,...,...,...,...,...,...,...
1084,2010,-0.861,0.043,-2.173,3.599,0.015,3.205160,2.969,Up
1085,2010,2.969,-0.861,0.043,-2.173,3.599,4.242568,1.281,Up
1086,2010,1.281,2.969,-0.861,0.043,-2.173,4.835082,0.283,Up
1087,2010,0.283,1.281,2.969,-0.861,0.043,4.454044,1.034,Up


# (a)

In [4]:
design = MS(["Lag1", "Lag2"])
X = design.fit_transform(weekly)
y = weekly["Direction"] == "Up"
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,0.2212,0.061,3.599,0.000
Lag1,-0.0387,0.026,-1.477,0.140
Lag2,0.0602,0.027,2.270,0.023


# (b)

In [5]:
design = MS(["Lag1", "Lag2"])
X = design.fit_transform(weekly.iloc[1:])
y = weekly.iloc[1:]["Direction"] == "Up"
model = sm.GLM(y, X, family=sm.families.Binomial())
results = model.fit()
summarize(results)

,coef,std err,z,P>|z|
intercept,0.2232,0.061,3.630,0.000
Lag1,-0.0384,0.026,-1.466,0.143
Lag2,0.0608,0.027,2.291,0.022


# (c)

In [16]:
X_test = design.transform(weekly.iloc[[0]])
prob = results.predict(X_test)

In [21]:
pred = "Up" if prob.iloc[0] > 0.5 else "Down"
pred == weekly.iloc[0]["Direction"]

False

# (d)

In [33]:
n = weekly.shape[0]
errors = []

for i in range(n):

    design = MS(["Lag1", "Lag2"])

    train = weekly.drop(index=i)
    test = weekly.iloc[[i]]

    X_train = design.fit_transform(train)
    y_train = train["Direction"] == "Up"

    model = sm.GLM(y_train, X_train, family=sm.families.Binomial())
    results = model.fit()

    X_test = design.transform(test)
    prob = results.predict(X_test) # since the design matrix is already fit we just need to transform using it

    pred = "Up" if prob.iloc[0] > 0.5 else "Down"
    actual = test["Direction"].iloc[0]

    error = int(pred != actual)

    errors.append(error)

# (e)

In [36]:
np.mean(errors)

np.float64(0.44995408631772266)